# t-SNE of MeshVAE latent representations

Each plotted point is one participant. The embedding is the deterministic latent mean (mu) produced by the trained scan-and-age VAE. t-SNE is fitted jointly to the training and held-out test embeddings for visualization only; it is not a predictive-performance metric.

In [ ]:
from __future__ import annotations

import gc
import inspect
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

REPO = Path('/data/projects/3D_scans/ML/MQVAE')
DATAROOT = Path('/data/projects/3D_scans/ML/MQVAE_combined_fast_20260808')
CHECKPOINTS = REPO / 'checkpoints'
EXPERIMENT = 'PLAQUEVOL_F_scanfix_20260815_vae_scan_age'
TRAIN_TSV = Path('/data/projects/3D_scans/datasets/3d_bodyscan_20260604_scanfix_20260811_train_F.tsv')
TEST_TSV = Path('/data/projects/3D_scans/datasets/3d_bodyscan_20260604_scanfix_20260811_test_F.tsv')
OUT_DIR = CHECKPOINTS / EXPERIMENT / 'tsne'

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

checkpoint = CHECKPOINTS / EXPERIMENT / 'best_vae_net.pth'
if not checkpoint.exists():
    raise FileNotFoundError(
        f'{checkpoint} does not exist yet. Wait for training to produce the best checkpoint.'
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', checkpoint)
print('Outputs:   ', OUT_DIR)

## Load the trained model

The checkpoint's cond_stats.json restores the exact encoder architecture, point-cloud settings, physical geometry channels, target definition, and target normalization used during training.

In [ ]:
from meshvae.conditioning import clean_stregkode, inverse_regression_target
from meshvae.datasets import PreprocessedMeshVAEDataset, mesh_collate_fn
from meshvae.model import MeshVAEModel
from meshvae.options import TestOptions

saved_argv = sys.argv[:]
sys.argv = [
    'tsne_plaque_latent',
    '--name', EXPERIMENT,
    '--dataroot', str(DATAROOT),
    '--checkpoints_dir', str(CHECKPOINTS),
    '--cond_tsv', str(TRAIN_TSV),
    '--feature_source', 'scan',
    '--only_gender', 'F',
    '--gpu_ids', '0',
    '--which_epoch', 'best',
    '--batch_size', '1',
]
try:
    opt = TestOptions().parse()
finally:
    sys.argv = saved_argv

# Build one dataset before the model, matching the diagnosis pipeline. This
# also restores and validates the physical scan-geometry input channels.
opt.cond_tsv = str(TRAIN_TSV)
initial_dataset = PreprocessedMeshVAEDataset(str(DATAROOT), opt)
model = MeshVAEModel(opt)
model.load_network('best')
model.eval()

net = model.net.module if isinstance(model.net, torch.nn.DataParallel) else model.net
net.eval()
print('Device:', model.device)
print('Latent dimensions:', net.latent_dim)
print('Regression transform:', opt.aux_regression_transform)

## Extract deterministic latent means

For the scan-and-age experiment, mu contains information available to the encoder: scan shape, restored physical geometry features, sex, and age. It never receives the plaque target itself.

In [ ]:
def encoder_condition(net, full_condition):
    if net.cond_dim <= 0 or net.encoder_cond_mode == 'none':
        return None
    if net.encoder_cond_mode == 'demo':
        return full_condition[:, :net.demo_cond_dim]
    raise ValueError(f'Unknown encoder condition mode: {net.encoder_cond_mode}')


@torch.no_grad()
def extract_dataset(dataset, split_name):
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=mesh_collate_fn,
    )
    records = []
    for index, batch in enumerate(loader, start=1):
        model.set_input(batch)
        enc_cond = encoder_condition(net, model.cond)
        _, mu = net.posterior(model.edge_features, model.mesh_batch, cond=enc_cond)

        pred_norm = net.regression_head(mu).reshape(-1).detach().cpu().numpy()
        pred_transformed = pred_norm * float(opt.aux_regression_std) + float(opt.aux_regression_mean)
        pred_raw = inverse_regression_target(pred_transformed, opt.aux_regression_transform)
        target_raw = batch['regression_target_raw'].detach().cpu().numpy().reshape(-1)
        latent = mu.detach().cpu().numpy()

        for path, target, prediction, vector in zip(batch['path'], target_raw, pred_raw, latent):
            row = {
                'participant_id': clean_stregkode(path) or str(path),
                'split': split_name,
                'total_plaque_volume': float(target),
                'predicted_total_plaque_volume': float(prediction),
            }
            row.update({f'latent_{i:03d}': float(value) for i, value in enumerate(vector)})
            records.append(row)

        if index % 100 == 0 or index == len(loader):
            print(f'[{split_name}] {index}/{len(loader)} ({100 * index / len(loader):.1f}%)')
    return pd.DataFrame(records)


train_latent = extract_dataset(initial_dataset, 'train')
del initial_dataset
gc.collect()

opt.cond_tsv = str(TEST_TSV)
test_dataset = PreprocessedMeshVAEDataset(str(DATAROOT), opt)
test_latent = extract_dataset(test_dataset, 'test')
del test_dataset
gc.collect()

latent_df = pd.concat([train_latent, test_latent], ignore_index=True)
latent_cols = [column for column in latent_df.columns if column.startswith('latent_')]
latent_df.to_csv(OUT_DIR / 'latent_embeddings.csv', index=False)
print(latent_df.groupby('split').size())
print('Latent table:', OUT_DIR / 'latent_embeddings.csv')

## Add age and calculate t-SNE

PCA reduces noise and computation before t-SNE. The fixed random seed makes the figure reproducible. The t-SNE coordinates themselves have no physical units.

In [ ]:
def numeric(series):
    return pd.to_numeric(
        series.astype(str).str.strip().str.strip('"').str.replace(',', '.', regex=False),
        errors='coerce',
    )


def metadata_for_split(path, split_name):
    frame = pd.read_csv(path, sep='\t', dtype=str)
    frame.columns = [str(column).strip().lstrip('\ufeff').strip('"') for column in frame.columns]
    id_col = 'participant_id' if 'participant_id' in frame.columns else 'stregkode'
    age_col = 'age' if 'age' in frame.columns else 'alder'
    agecut_col = 'agecut' if 'agecut' in frame.columns else None
    result = pd.DataFrame({
        'participant_id': frame[id_col].map(clean_stregkode),
        'age': numeric(frame[age_col]),
        'split': split_name,
    })
    result['age_group'] = frame[agecut_col].astype(str).str.strip().str.strip('"') if agecut_col else pd.NA
    plaque_cols = [
        'plaque_volume_right_carotid_corrected',
        'plaque_volume_left_carotid_corrected',
        'plaque_volume_right_femoral_corrected',
        'plaque_volume_left_femoral_corrected',
    ]
    if all(column in frame.columns for column in plaque_cols):
        plaque_values = pd.concat([numeric(frame[column]).rename(column) for column in plaque_cols], axis=1)
        result['metadata_total_plaque_volume'] = plaque_values.sum(axis=1, min_count=len(plaque_cols))
    else:
        result['metadata_total_plaque_volume'] = np.nan
    return result.dropna(subset=['participant_id']).drop_duplicates(['participant_id', 'split'])


metadata = pd.concat(
    [metadata_for_split(TRAIN_TSV, 'train'), metadata_for_split(TEST_TSV, 'test')],
    ignore_index=True,
)
latent_df = latent_df.merge(metadata, on=['participant_id', 'split'], how='left', validate='many_to_one')

X = latent_df[latent_cols].to_numpy(dtype=np.float64)
finite_rows = np.isfinite(X).all(axis=1)
if not finite_rows.all():
    print(f'Dropping {(~finite_rows).sum()} rows with non-finite latent values')
    latent_df = latent_df.loc[finite_rows].reset_index(drop=True)
    X = X[finite_rows]

variable_dimensions = np.var(X, axis=0) > 1e-10
X = X[:, variable_dimensions]
X_scaled = StandardScaler().fit_transform(X)
pca_dimensions = min(50, X_scaled.shape[1], len(X_scaled) - 1)
X_pca = PCA(n_components=pca_dimensions, random_state=42).fit_transform(X_scaled)

perplexity = min(30.0, max(5.0, (len(X_pca) - 1) / 3.0))
tsne_kwargs = dict(
    n_components=2,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=42,
    verbose=1,
)
if 'max_iter' in inspect.signature(TSNE).parameters:
    tsne_kwargs['max_iter'] = 2000
else:
    tsne_kwargs['n_iter'] = 2000

coords = TSNE(**tsne_kwargs).fit_transform(X_pca)
latent_df['tsne_1'] = coords[:, 0]
latent_df['tsne_2'] = coords[:, 1]
latent_df['plaque_present'] = np.where(latent_df['total_plaque_volume'] > 0, 'Plaque > 0', 'Plaque = 0')
latent_df.to_csv(OUT_DIR / 'tsne_coordinates.csv', index=False)

print(f'n={len(latent_df)}, retained_latent_dims={X.shape[1]}, PCA_dims={pca_dimensions}, perplexity={perplexity:.1f}')
print('Coordinates:', OUT_DIR / 'tsne_coordinates.csv')

## Plot four complementary views

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 11), constrained_layout=True)
x = latent_df['tsne_1'].to_numpy()
y = latent_df['tsne_2'].to_numpy()

plaque_color = np.log1p(latent_df['total_plaque_volume'].to_numpy())
scatter = axes[0, 0].scatter(x, y, c=plaque_color, cmap='viridis', s=18, alpha=0.78, linewidths=0)
fig.colorbar(scatter, ax=axes[0, 0], label='log1p(total corrected plaque volume)')
axes[0, 0].set_title('True total plaque burden')

age_valid = latent_df['age'].notna().to_numpy()
scatter = axes[0, 1].scatter(x[age_valid], y[age_valid], c=latent_df.loc[age_valid, 'age'], cmap='plasma', s=18, alpha=0.78, linewidths=0)
fig.colorbar(scatter, ax=axes[0, 1], label='Age (years)')
axes[0, 1].set_title('Age')

presence_colors = {'Plaque = 0': '#2878B5', 'Plaque > 0': '#D94841'}
for label, color in presence_colors.items():
    mask = latent_df['plaque_present'].eq(label).to_numpy()
    axes[1, 0].scatter(x[mask], y[mask], c=color, label=f'{label} (n={mask.sum()})', s=18, alpha=0.72, linewidths=0)
axes[1, 0].legend(frameon=True)
axes[1, 0].set_title('Plaque presence')

split_styles = {'train': ('#2D6A4F', 'o'), 'test': ('#7B2CBF', '^')}
for label, (color, marker) in split_styles.items():
    mask = latent_df['split'].eq(label).to_numpy()
    axes[1, 1].scatter(x[mask], y[mask], c=color, marker=marker, label=f'{label} (n={mask.sum()})', s=20, alpha=0.72, linewidths=0)
axes[1, 1].legend(frameon=True)
axes[1, 1].set_title('Train versus held-out test')

for ax in axes.flat:
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f'MeshVAE latent space: {EXPERIMENT}', fontsize=15)
png_path = OUT_DIR / 'tsne_latent_plots.png'
pdf_path = OUT_DIR / 'tsne_latent_plots.pdf'
fig.savefig(png_path, dpi=220, bbox_inches='tight')
fig.savefig(pdf_path, bbox_inches='tight')
plt.show()
print('Figure:', png_path)
print('PDF:   ', pdf_path)

## Interpretation cautions

- Nearby points have similar learned representations; the absolute axes, orientation, and global distances have no direct meaning.
- A smooth plaque-color gradient would indicate that plaque burden is organized in the latent space. Complete class separation is not expected for a continuous biological outcome.
- A strong age gradient is expected because this model explicitly includes age in the encoder. Use the scan-only checkpoint for a t-SNE intended to isolate scan morphology.
- Train and test points should broadly overlap. A separate test island can indicate a cohort or preprocessing shift.
- t-SNE can create apparent clusters even in continuous data. Treat it as exploratory visualization, not evidence of diagnostic accuracy or biological subtypes.

## Scan-only latent space

This final cell repeats the analysis using the separately trained scan-only plaque-volume model. Age is still used to color one panel after extraction, but it is not supplied to the encoder. This distinction is essential: removing an age column from a plot cannot remove age information that was already used when the original embedding was created.

In [ ]:
SCAN_ONLY_EXPERIMENT = 'FULL15_F_scanfix_20260811_vae_scan_noage'
SCAN_ONLY_OUT_DIR = CHECKPOINTS / SCAN_ONLY_EXPERIMENT / 'tsne'
scan_only_checkpoint = CHECKPOINTS / SCAN_ONLY_EXPERIMENT / 'best_vae_net.pth'
if not scan_only_checkpoint.exists():
    raise FileNotFoundError(
        f'{scan_only_checkpoint} does not exist. Check the name prefix used for the completed 15-model run.'
    )
SCAN_ONLY_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reload the completed atherosclerosis model's scan-only checkpoint. Plaque
# volume is attached from the TSV after latent extraction; this checkpoint
# correctly has no continuous-regression head.
saved_argv = sys.argv[:]
sys.argv = [
    'tsne_plaque_scan_only',
    '--name', SCAN_ONLY_EXPERIMENT,
    '--dataroot', str(DATAROOT),
    '--checkpoints_dir', str(CHECKPOINTS),
    '--cond_tsv', str(TRAIN_TSV),
    '--feature_source', 'scan',
    '--only_gender', 'F',
    '--gpu_ids', '0',
    '--which_epoch', 'best',
    '--batch_size', '1',
]
try:
    opt = TestOptions().parse()
finally:
    sys.argv = saved_argv

opt.cond_tsv = str(TRAIN_TSV)
scan_train_dataset = PreprocessedMeshVAEDataset(str(DATAROOT), opt)
model = MeshVAEModel(opt)
model.load_network('best')
model.eval()
net = model.net.module if isinstance(model.net, torch.nn.DataParallel) else model.net
net.eval()
assert net.encoder_cond_mode == 'none', f'Expected scan-only encoder, got {net.encoder_cond_mode}'

@torch.no_grad()
def extract_latent_only(dataset, split_name):
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=mesh_collate_fn)
    records = []
    for index, batch in enumerate(loader, start=1):
        model.set_input(batch)
        enc_cond = encoder_condition(net, model.cond)
        _, mu = net.posterior(model.edge_features, model.mesh_batch, cond=enc_cond)
        for path, vector in zip(batch['path'], mu.detach().cpu().numpy()):
            row = {'participant_id': clean_stregkode(path) or str(path), 'split': split_name}
            row.update({f'latent_{i:03d}': float(value) for i, value in enumerate(vector)})
            records.append(row)
        if index % 100 == 0 or index == len(loader):
            print(f'[{split_name}] {index}/{len(loader)} ({100 * index / len(loader):.1f}%)')
    return pd.DataFrame(records)


scan_train_latent = extract_latent_only(scan_train_dataset, 'train')
del scan_train_dataset
gc.collect()

opt.cond_tsv = str(TEST_TSV)
scan_test_dataset = PreprocessedMeshVAEDataset(str(DATAROOT), opt)
scan_test_latent = extract_latent_only(scan_test_dataset, 'test')
del scan_test_dataset
gc.collect()

scan_df = pd.concat([scan_train_latent, scan_test_latent], ignore_index=True)
scan_latent_cols = [column for column in scan_df.columns if column.startswith('latent_')]
scan_df = scan_df.merge(metadata, on=['participant_id', 'split'], how='left', validate='many_to_one')
scan_df['total_plaque_volume'] = scan_df['metadata_total_plaque_volume']
before_complete_plaque = len(scan_df)
scan_df = scan_df.dropna(subset=['total_plaque_volume']).reset_index(drop=True)
print(f'[scan-only] retained {len(scan_df)}/{before_complete_plaque} participants with all four plaque-volume measurements')
scan_df.to_csv(SCAN_ONLY_OUT_DIR / 'latent_embeddings.csv', index=False)

scan_X = scan_df[scan_latent_cols].to_numpy(dtype=np.float64)
scan_finite = np.isfinite(scan_X).all(axis=1)
scan_df = scan_df.loc[scan_finite].reset_index(drop=True)
scan_X = scan_X[scan_finite]
scan_variable = np.var(scan_X, axis=0) > 1e-10
scan_X = StandardScaler().fit_transform(scan_X[:, scan_variable])
scan_pca_dimensions = min(50, scan_X.shape[1], len(scan_X) - 1)
scan_X = PCA(n_components=scan_pca_dimensions, random_state=42).fit_transform(scan_X)
scan_perplexity = min(30.0, max(5.0, (len(scan_X) - 1) / 3.0))

scan_tsne_kwargs = dict(
    n_components=2,
    perplexity=scan_perplexity,
    init='pca',
    learning_rate='auto',
    random_state=42,
    verbose=1,
)
if 'max_iter' in inspect.signature(TSNE).parameters:
    scan_tsne_kwargs['max_iter'] = 2000
else:
    scan_tsne_kwargs['n_iter'] = 2000
scan_coords = TSNE(**scan_tsne_kwargs).fit_transform(scan_X)
scan_df['tsne_1'] = scan_coords[:, 0]
scan_df['tsne_2'] = scan_coords[:, 1]
scan_df['plaque_present'] = np.where(scan_df['total_plaque_volume'] > 0, 'Plaque > 0', 'Plaque = 0')
scan_df.to_csv(SCAN_ONLY_OUT_DIR / 'tsne_coordinates.csv', index=False)

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 11), constrained_layout=True)
scan_x = scan_df['tsne_1'].to_numpy()
scan_y = scan_df['tsne_2'].to_numpy()

plaque_color = np.log1p(scan_df['total_plaque_volume'].to_numpy())
scatter = axes[0, 0].scatter(scan_x, scan_y, c=plaque_color, cmap='viridis', s=18, alpha=0.78, linewidths=0)
fig.colorbar(scatter, ax=axes[0, 0], label='log1p(total corrected plaque volume)')
axes[0, 0].set_title('True total plaque burden')

age_valid = scan_df['age'].notna().to_numpy()
scatter = axes[0, 1].scatter(scan_x[age_valid], scan_y[age_valid], c=scan_df.loc[age_valid, 'age'], cmap='plasma', s=18, alpha=0.78, linewidths=0)
fig.colorbar(scatter, ax=axes[0, 1], label='Age (years)')
axes[0, 1].set_title('Age, shown only as an external color')

presence_colors = {'Plaque = 0': '#2878B5', 'Plaque > 0': '#D94841'}
for label, color in presence_colors.items():
    mask = scan_df['plaque_present'].eq(label).to_numpy()
    axes[1, 0].scatter(scan_x[mask], scan_y[mask], c=color, label=f'{label} (n={mask.sum()})', s=18, alpha=0.72, linewidths=0)
axes[1, 0].legend(frameon=True)
axes[1, 0].set_title('Plaque presence')

split_styles = {'train': ('#2D6A4F', 'o'), 'test': ('#7B2CBF', '^')}
for label, (color, marker) in split_styles.items():
    mask = scan_df['split'].eq(label).to_numpy()
    axes[1, 1].scatter(scan_x[mask], scan_y[mask], c=color, marker=marker, label=f'{label} (n={mask.sum()})', s=20, alpha=0.72, linewidths=0)
axes[1, 1].legend(frameon=True)
axes[1, 1].set_title('Train versus held-out test')

for ax in axes.flat:
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f'Scan-only MeshVAE latent space: {SCAN_ONLY_EXPERIMENT}', fontsize=15)
scan_png = SCAN_ONLY_OUT_DIR / 'tsne_scan_only_plots.png'
scan_pdf = SCAN_ONLY_OUT_DIR / 'tsne_scan_only_plots.pdf'
fig.savefig(scan_png, dpi=220, bbox_inches='tight')
fig.savefig(scan_pdf, bbox_inches='tight')
plt.show()

print(f'n={len(scan_df)}, PCA_dims={scan_pca_dimensions}, perplexity={scan_perplexity:.1f}')
print('Figure:', scan_png)
print('Coordinates:', SCAN_ONLY_OUT_DIR / 'tsne_coordinates.csv')

## Attach body measurements to the fixed scan-only map

This cell merges body measurements from the same train and test TSV files. It does not rerun t-SNE, so every participant remains at exactly the same coordinates as in the scan-only figure above.

In [ ]:
MEASUREMENT_COLUMNS = {
    'body_mass_index': ['body_mass_index', 'bmi'],
    'waist_circumference_cm': ['waist_circumference_cm', 'waist_cm'],
    'hip_circumference_cm': ['hip_circumference_cm', 'hip_cm'],
    'height_cm': ['height_cm', 'height'],
    'weight_kg': ['weight_kg', 'weight'],
}


def find_column(frame, candidates):
    lower_to_actual = {str(column).lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lower_to_actual:
            return lower_to_actual[candidate.lower()]
    return None


def body_measurements_for_split(path, split_name):
    frame = pd.read_csv(path, sep='\t', dtype=str)
    frame.columns = [str(column).strip().lstrip('\ufeff').strip('"') for column in frame.columns]
    id_col = find_column(frame, ['participant_id', 'stregkode', 'participantid'])
    if id_col is None:
        raise KeyError(f'No participant ID column found in {path}')

    result = pd.DataFrame({
        'participant_id': frame[id_col].map(clean_stregkode),
        'split': split_name,
    })
    resolved = {}
    for output_name, candidates in MEASUREMENT_COLUMNS.items():
        source_col = find_column(frame, candidates)
        resolved[output_name] = source_col
        result[output_name] = numeric(frame[source_col]) if source_col else np.nan

    print(f'[{split_name}] measurement columns:', resolved)
    return result.dropna(subset=['participant_id']).drop_duplicates(['participant_id', 'split'])


body_metadata = pd.concat(
    [
        body_measurements_for_split(TRAIN_TSV, 'train'),
        body_measurements_for_split(TEST_TSV, 'test'),
    ],
    ignore_index=True,
)

# Make this cell safe to rerun without creating _x and _y duplicate columns.
measurement_names = list(MEASUREMENT_COLUMNS)
scan_df = scan_df.drop(columns=[column for column in measurement_names + ['waist_to_hip_ratio'] if column in scan_df.columns])
scan_df = scan_df.merge(body_metadata, on=['participant_id', 'split'], how='left', validate='many_to_one')
scan_df['waist_to_hip_ratio'] = np.where(
    scan_df['hip_circumference_cm'] > 0,
    scan_df['waist_circumference_cm'] / scan_df['hip_circumference_cm'],
    np.nan,
)

summary_columns = measurement_names + ['waist_to_hip_ratio']
measurement_summary = pd.DataFrame({
    'available_n': scan_df[summary_columns].notna().sum(),
    'missing_n': scan_df[summary_columns].isna().sum(),
    'median': scan_df[summary_columns].median(),
    'minimum': scan_df[summary_columns].min(),
    'maximum': scan_df[summary_columns].max(),
})
display(measurement_summary)
scan_df.to_csv(SCAN_ONLY_OUT_DIR / 'tsne_coordinates_with_body_measurements.csv', index=False)
print('Enriched coordinates:', SCAN_ONLY_OUT_DIR / 'tsne_coordinates_with_body_measurements.csv')

## Color the same coordinates by body measurements

Color limits use the 2nd and 98th percentiles so a few extreme observations do not hide the pattern for everyone else. Participants with missing values are shown in light gray.

In [ ]:
body_panels = [
    ('body_mass_index', 'Body mass index (kg/m^2)', 'magma'),
    ('waist_circumference_cm', 'Waist circumference (cm)', 'viridis'),
    ('hip_circumference_cm', 'Hip circumference (cm)', 'cividis'),
    ('height_cm', 'Height (cm)', 'plasma'),
    ('weight_kg', 'Weight (kg)', 'inferno'),
    ('waist_to_hip_ratio', 'Waist-to-hip ratio', 'turbo'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
fixed_x = scan_df['tsne_1'].to_numpy(dtype=float)
fixed_y = scan_df['tsne_2'].to_numpy(dtype=float)

for ax, (column, title, cmap) in zip(axes.flat, body_panels):
    values = scan_df[column].to_numpy(dtype=float)
    valid = np.isfinite(values)
    ax.scatter(fixed_x[~valid], fixed_y[~valid], c='#D0D0D0', s=15, alpha=0.45, linewidths=0)

    if valid.any():
        vmin, vmax = np.nanquantile(values[valid], [0.02, 0.98])
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = np.nanmin(values[valid]), np.nanmax(values[valid])
        points = ax.scatter(
            fixed_x[valid],
            fixed_y[valid],
            c=values[valid],
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            s=18,
            alpha=0.78,
            linewidths=0,
        )
        fig.colorbar(points, ax=ax, label=title, extend='both')

    ax.set_title(f'{title} (n={valid.sum()})')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Scan-only MeshVAE latent space colored by body measurements', fontsize=16)
body_png = SCAN_ONLY_OUT_DIR / 'tsne_scan_only_body_measurements.png'
body_pdf = SCAN_ONLY_OUT_DIR / 'tsne_scan_only_body_measurements.pdf'
fig.savefig(body_png, dpi=220, bbox_inches='tight')
fig.savefig(body_pdf, bbox_inches='tight')
plt.show()
print('Figure:', body_png)
print('PDF:   ', body_pdf)

### How to read these panels

Look for smooth color changes along branches rather than perfectly separated color groups. A smooth gradient indicates that the corresponding body measurement helps organize the learned scan representation. Similar patterns in BMI, waist, and weight are expected because those measurements are correlated. The waist-to-hip panel can help distinguish general body size from central fat distribution. These plots remain exploratory; the curved t-SNE geometry and distances between remote regions are not quantitative effect sizes.

## Quantify body measurements from the original latent vectors

This analysis deliberately uses the original high-dimensional latent vectors rather than the two t-SNE coordinates. For each body measurement, a Ridge regression is tuned by five-fold cross-validation using training participants only and is then evaluated once on the held-out test participants.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline

ridge_targets = [
    ('body_mass_index', 'BMI'),
    ('waist_circumference_cm', 'Waist circumference'),
    ('hip_circumference_cm', 'Hip circumference'),
    ('height_cm', 'Height'),
    ('weight_kg', 'Weight'),
    ('waist_to_hip_ratio', 'Waist-to-hip ratio'),
]
alpha_grid = np.logspace(-4, 4, 17)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
r2_rows = []
prediction_rows = []

for target_col, target_label in ridge_targets:
    train_mask = scan_df['split'].eq('train') & scan_df[target_col].notna()
    test_mask = scan_df['split'].eq('test') & scan_df[target_col].notna()
    X_train = scan_df.loc[train_mask, scan_latent_cols].to_numpy(dtype=float)
    y_train = scan_df.loc[train_mask, target_col].to_numpy(dtype=float)
    X_test = scan_df.loc[test_mask, scan_latent_cols].to_numpy(dtype=float)
    y_test = scan_df.loc[test_mask, target_col].to_numpy(dtype=float)

    estimator = Pipeline([
        ('scale', StandardScaler()),
        ('ridge', Ridge()),
    ])
    search = GridSearchCV(
        estimator,
        param_grid={'ridge__alpha': alpha_grid},
        scoring='r2',
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    prediction = search.predict(X_test)
    pearson = (
        float(np.corrcoef(y_test, prediction)[0, 1])
        if len(y_test) > 1 and np.std(y_test) > 0 and np.std(prediction) > 0
        else np.nan
    )
    r2_rows.append({
        'Measurement': target_label,
        'N_train': len(y_train),
        'N_test': len(y_test),
        'CV_R2_train_only': search.best_score_,
        'Test_R2': r2_score(y_test, prediction),
        'Test_MAE': mean_absolute_error(y_test, prediction),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, prediction)),
        'Test_Pearson': pearson,
        'Ridge_alpha': search.best_params_['ridge__alpha'],
    })
    for participant_id, observed, predicted in zip(
        scan_df.loc[test_mask, 'participant_id'], y_test, prediction
    ):
        prediction_rows.append({
            'participant_id': participant_id,
            'measurement': target_col,
            'observed': observed,
            'predicted': predicted,
        })

latent_r2_results = pd.DataFrame(r2_rows).sort_values('Test_R2', ascending=False).reset_index(drop=True)
display(latent_r2_results.style.format({
    'CV_R2_train_only': '{:.3f}',
    'Test_R2': '{:.3f}',
    'Test_MAE': '{:.3f}',
    'Test_RMSE': '{:.3f}',
    'Test_Pearson': '{:.3f}',
    'Ridge_alpha': '{:.4g}',
}))

r2_path = SCAN_ONLY_OUT_DIR / 'latent_body_measurement_r2.csv'
prediction_path = SCAN_ONLY_OUT_DIR / 'latent_body_measurement_test_predictions.csv'
latent_r2_results.to_csv(r2_path, index=False)
pd.DataFrame(prediction_rows).to_csv(prediction_path, index=False)
print('R2 summary:', r2_path)
print('Held-out predictions:', prediction_path)